# Day 24 — Pomo Capstone: CLI + Ship

> ⚠️ **Why this matters.** Yesterday you built the storage; today you wrap it in a CLI with argparse, write the tests, and ship the whole thing. By tonight `pomo --help` works on your machine.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/curriculum/blob/main/phase-1-python-cli/lessons/24-pomo-capstone-ship.ipynb)

## What you'll do today

- [ ] Wire `cli.py` with argparse: start, stop, list, report
- [ ] Write tests (unit + integration)
- [ ] Package: `pyproject.toml`, README, `uv tool install`
- [ ] Demo to your mentor (Friday call)

## 1. CLI skeleton

In [ ]:
# src/pomo/cli.py
import argparse
from pathlib import Path
from pomo.storage import Storage

DB_PATH = Path.home() / '.pomo' / 'pomo.db'

def cmd_start(args):
    storage = Storage(DB_PATH)
    sid = storage.start(args.duration, args.tag)
    print(f'Started session #{sid} ({args.duration} min, tag={args.tag or "none"})')

def cmd_stop(args):
    storage = Storage(DB_PATH)
    sid = storage.stop()
    print(f'Stopped session #{sid}')

def cmd_list(args):
    storage = Storage(DB_PATH)
    for row in storage.list(args.since):
        print(f"#{row['id']} {row['started_at']} {row['duration_min']}m {row['tag'] or '-'}")

def cmd_report(args):
    storage = Storage(DB_PATH)
    for row in storage.report():
        print(f"{row['tag'] or '(no tag)':20} {row['total']} min")

def main():
    parser = argparse.ArgumentParser(prog='pomo', description='Pomodoro tracker.')
    sub = parser.add_subparsers(dest='cmd', required=True)
    
    p_start = sub.add_parser('start')
    p_start.add_argument('--duration', type=int, default=25, help='minutes (default 25)')
    p_start.add_argument('--tag', default=None)
    p_start.set_defaults(func=cmd_start)
    
    sub.add_parser('stop').set_defaults(func=cmd_stop)
    
    p_list = sub.add_parser('list')
    p_list.add_argument('--since', default=None, help='YYYY-MM-DD')
    p_list.set_defaults(func=cmd_list)
    
    sub.add_parser('report').set_defaults(func=cmd_report)
    
    args = parser.parse_args()
    try:
        args.func(args)
    except ValueError as e:
        print(f'Error: {e}', flush=True)
        raise SystemExit(1)

if __name__ == '__main__':
    main()

## 2. Tests for the CLI layer

CLI integration tests are tricky — they involve sys.argv and capsys. Easier pattern: extract the logic into testable functions, keep the CLI as a thin wrapper.

In [ ]:
# tests/test_cli.py
from pomo.cli import cmd_start, cmd_stop, cmd_list
from argparse import Namespace

def test_start_creates_session(tmp_path, monkeypatch):
    import pomo.cli
    monkeypatch.setattr(pomo.cli, 'DB_PATH', tmp_path / 'pomo.db')
    args = Namespace(duration=25, tag='phase-1')
    cmd_start(args)
    # verify it landed in the DB
    from pomo.storage import Storage
    storage = Storage(tmp_path / 'pomo.db')
    rows = storage.list()
    assert len(rows) == 1
    assert rows[0]['tag'] == 'phase-1'

**Or use subprocess** to test end-to-end:

In [ ]:
# tests/test_end_to_end.py
import subprocess, json

def test_install_works():
    result = subprocess.run(['pomo', '--help'], capture_output=True, text=True)
    assert result.returncode == 0
    assert 'Pomodoro' in result.stdout

## 3. The final pyproject.toml

```toml
[project]
name = 'pomo'
version = '0.1.0'
description = 'A simple Pomodoro tracker CLI'
readme = 'README.md'
requires-python = '>=3.13'
dependencies = []

[project.scripts]
pomo = 'pomo.cli:main'

[build-system]
requires = ['hatchling']
build-backend = 'hatchling.build'

[tool.hatch.build.targets.wheel]
packages = ['src/pomo']
```

No deps! sqlite3 + argparse + datetime are all stdlib. Tiny tool, zero install pain.

## 4. Install & demo

```bash
uv tool install --from . pomo

pomo start --duration 1 --tag demo
# wait 70 seconds
pomo stop
pomo list
pomo report
```

Friday demo: walk the mentor through. Show:

- The repo + git log (clean commits)
- CI green
- Coverage report
- `pomo --help` and a few commands
- README explaining install/usage
- v0.1.0 tag

## End-of-day acceptance

Everything below must work on a **fresh machine** (or fresh `uv tool uninstall pomo` first):

1. `git clone YOUR_REPO && cd pomo`
2. `uv sync` — no errors
3. `uv run pytest --cov=src/pomo` — green, ≥80% coverage
4. `uv run ruff check .` and `uv run mypy src` — clean
5. `uv tool install --from . pomo` — installs
6. `pomo start --tag demo` then `pomo stop` then `pomo report` — all work
7. CI badge on README is green
8. v0.1.0 git tag pushed

## Connect to the project

> 🎯 **End of Phase 1 — tomorrow (Day 25):** Phase 1 retrospective. You'll write up what you built, what you'd do differently, and demo both `english-helper` AND `pomo` to your mentor for the gate.

**Quiz:** [24-pomo-capstone-ship-quiz.ipynb](24-pomo-capstone-ship-quiz.ipynb)